In [29]:
import os
import json
import warnings
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import tensorflow as tf
from scipy.stats import wilcoxon, norm
from scipy.stats import rankdata
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    classification_report, cohen_kappa_score
)
import h5py

In [30]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [31]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s-%(levelname)s-%(message)s',
    datefmt='%H:%M:%S')
log = logging.getLogger(__name__)
    

In [32]:
CFG = {
    "data_path":    "../data/processed/glioma_dataset.h5",
    "split_file":   "../data/splits/manual_splits.json",
    "model_a_dir":  "../results/models/baselines",   
    "model_b_dir":  "../results/models/tstr",       
    "output_dir":   "../results/statistical_eval",


    "n_bootstrap":  10_000,    
    "alpha":        0.05,      
    "random_seed":  42,

    # Modele evaluate
    "models": {
        "Apollo":      {"prefix": "apollo", "ext": ".joblib", "type": "sklearn"},
        "RF_Standard": {"prefix": "rf",     "ext": ".joblib", "type": "sklearn"},
        "XGBoost":     {"prefix": "xgb",    "ext": ".json",   "type": "xgboost"},
        "1D_CNN":      {"prefix": "cnn",    "ext": ".keras",  "type": "keras"},
    },

    "primary_metric": "f1_macro",
}

np.random.seed(CFG["random_seed"])


In [ ]:
def find_model_path(base_dir: str, prefix: str, ext: str, fold_idx: int) -> str:
    candidates = [
        f"{base_dir}/{prefix}_fold_{fold_idx}{ext}",
        f"{base_dir}/{prefix}_fold{fold_idx}{ext}",
        f"{base_dir}/{prefix}_fold-{fold_idx}{ext}",
        f"{base_dir}/{prefix}_{fold_idx}{ext}",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        f"Model '{prefix}' fold {fold_idx} negăsit. Căi încercate:\n"
        + "\n".join(f"  {c}" for c in candidates)
    )


def compute_metric(y_true: np.ndarray, y_pred: np.ndarray, metric: str) -> float:
    """Calculează metrica."""
    if metric == "accuracy":
        return accuracy_score(y_true, y_pred)
    elif metric == "f1_macro":
        return f1_score(y_true, y_pred, average="macro", zero_division=0)
    elif metric == "balanced_accuracy":
        return balanced_accuracy_score(y_true, y_pred)
    else:
        raise ValueError(f"Metrică necunoscută: '{metric}'")


def bootstrap_ci(
    values: np.ndarray,
    stat_fn=np.mean,
    n_boot: int = 10_000,
    alpha: float = 0.05,
    seed: int = 42,
) -> tuple[float, float, float]:
    """
    Bootstrap percentile CI pentru o statistică arbitrară.

    Returns
    -------
    (point_estimate, lower_bound, upper_bound)
    """
    rng = np.random.default_rng(seed)
    boot_stats = np.array([
        stat_fn(rng.choice(values, size=len(values), replace=True))
        for _ in range(n_boot)
    ])
    lo = np.percentile(boot_stats, 100 * alpha / 2)
    hi = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return stat_fn(values), lo, hi


def wilcoxon_effect_size(stat_z: float, n_pairs: int) -> float:
    """
    Effect size r = Z / sqrt(N) pentru testul Wilcoxon signed-rank.
    Interpretare: 0.1 mic | 0.3 moderat | 0.5+ mare
    """
    return abs(stat_z) / np.sqrt(n_pairs)


def interpret_effect_size(r: float) -> str:
    if r < 0.1:
        return "neglijabil"
    elif r < 0.3:
        return "mic"
    elif r < 0.5:
        return "moderat"
    else:
        return "mare"


def wilcoxon_z_from_stat(stat_w: float, diffs: np.ndarray) -> float:
    """
    Convertește statistica W Wilcoxon în z-score standardizat,
    folosind aproximarea normală cu corecție pentru ties.
    """
    n = len(diffs)
    # Ranguri pe valorile absolute ale diferențelor non-zero
    nonzero = diffs[diffs != 0]
    m = len(nonzero)
    if m == 0:
        return 0.0

    ranks = rankdata(np.abs(nonzero))

    # Tie correction
    unique_ranks, counts = np.unique(ranks, return_counts=True)
    tie_sum = np.sum(counts ** 3 - counts)

    # Varianță cu corecție ties
    var_w = (m * (m + 1) * (2 * m + 1) / 6) - (tie_sum / 48)
    mean_w = m * (m + 1) / 4

    if var_w <= 0:
        return 0.0

    z = (stat_w - mean_w) / np.sqrt(var_w)
    return z


def format_pvalue(p: float) -> str:
    """Formatare p-value."""
    if p < 0.001:
        return "p < 0.001"
    elif p < 0.01:
        return f"p = {p:.3f}"
    else:
        return f"p = {p:.4f}"


def print_section(title: str, width: int = 70):
    print(f"\n{'═' * width}")
    print(f"  {title}")
    print(f"{'═' * width}")


def print_subsection(title: str, width: int = 60):
    print(f"\n  {'─' * width}")
    print(f"  ▸ {title}")
    print(f"  {'─' * width}")



#  PREDICȚIE MODELE
def _load_keras_model_safe(path: str):

    import zipfile, tempfile, shutil, json as _json

   
    try:
        return tf.keras.models.load_model(path, compile=False)
    except Exception as e_direct:
        log.debug(f"Încărcare directă eșuată ({type(e_direct).__name__}), aplicăm patch...")

    KEYS_TO_STRIP = {"quantization_config"}   

    def _strip_config(obj):
        
        if isinstance(obj, dict):
            return {
                k: _strip_config(v)
                for k, v in obj.items()
                if k not in KEYS_TO_STRIP
            }
        elif isinstance(obj, list):
            return [_strip_config(item) for item in obj]
        return obj

    tmp_dir = tempfile.mkdtemp(prefix="keras_patch_")
    try:

        
        if zipfile.is_zipfile(path):
            tmp_model_dir = os.path.join(tmp_dir, "model_patched")
            os.makedirs(tmp_model_dir, exist_ok=True)

            with zipfile.ZipFile(path, "r") as zin:
                zin.extractall(tmp_model_dir)

            config_path = os.path.join(tmp_model_dir, "config.json")
            if os.path.exists(config_path):
                with open(config_path, "r", encoding="utf-8") as f:
                    cfg_obj = _json.load(f)

                cfg_patched = _strip_config(cfg_obj)

                with open(config_path, "w", encoding="utf-8") as f:
                    _json.dump(cfg_patched, f, ensure_ascii=False)

                log.debug("config.json patch aplicat (ZIP .keras).")

            
            tmp_keras_path = os.path.join(tmp_dir, "model_patched.keras")
            with zipfile.ZipFile(tmp_keras_path, "w", zipfile.ZIP_DEFLATED) as zout:
                for root, _, files in os.walk(tmp_model_dir):
                    for file in files:
                        abs_path = os.path.join(root, file)
                        arc_name = os.path.relpath(abs_path, tmp_model_dir)
                        zout.write(abs_path, arc_name)

            return tf.keras.models.load_model(tmp_keras_path, compile=False)

        
        else:
            tmp_h5_path = os.path.join(tmp_dir, "model_patched.h5")
            shutil.copy2(path, tmp_h5_path)

            with h5py.File(tmp_h5_path, "r+") as f:
                for attr_name in ("model_config", "keras_model_config"):
                    if attr_name in f.attrs:
                        raw = f.attrs[attr_name]
                        cfg_str = raw.decode("utf-8") if isinstance(raw, bytes) else raw
                        cfg_obj = _json.loads(cfg_str)
                        cfg_patched = _strip_config(cfg_obj)
                        f.attrs[attr_name] = _json.dumps(
                            cfg_patched, ensure_ascii=False
                        ).encode("utf-8")
                        log.debug(f"Atribut HDF5 '{attr_name}' patch aplicat.")

            return tf.keras.models.load_model(tmp_h5_path, compile=False)

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def load_and_predict(
    path: str,
    model_type: str,
    X: np.ndarray,
) -> np.ndarray:
    """
    Încarcă modelul și returnează predicțiile.
    Suportă sklearn (joblib), XGBoost (.json) și Keras (.keras / .h5).
    """
    if model_type == "sklearn":
        model = joblib.load(path)
        return model.predict(X)

    elif model_type == "xgboost":
        model = xgb.XGBClassifier()
        model.load_model(path)
        return model.predict(X)

    elif model_type == "keras":
        X_3d = X.reshape(-1, X.shape[1], 1)
        model = _load_keras_model_safe(path)
        preds = np.argmax(model.predict(X_3d, verbose=0), axis=1)
        tf.keras.backend.clear_session()
        return preds


    else:
        raise ValueError(f"Tip model necunoscut: '{model_type}'")



In [34]:
#  EVALUARE PRINCIPALĂ


def evaluate_model(
    model_name: str,
    cfg: dict,
    X_all: np.ndarray,
    y_all: np.ndarray,
    samples_all: np.ndarray,
    splits: dict,
) -> dict:
    """
    Evaluează un singur model (A vs B) pe toate fold-urile și returnează
    un dicționar complet cu statistici.
    """
    m_cfg = cfg["models"][model_name]
    prefix = m_cfg["prefix"]
    ext    = m_cfg["ext"]
    mtype  = m_cfg["type"]
    metric = cfg["primary_metric"]

    log.info(f"Evaluare model: {model_name}")

    scores_A, scores_B = [], []
    preds_A_all, preds_B_all, y_true_all = [], [], []

    # Iterăm pe fold-uri 
    n_folds = len(splits)
    for fold_idx in range(n_folds):
        fold_key = (
            str(fold_idx) if str(fold_idx) in splits
            else f"fold_{fold_idx}"
        )
        val_samples = splits[fold_key]["test"]
        val_mask    = np.isin(samples_all, val_samples)
        X_val       = X_all[val_mask]
        y_val       = y_all[val_mask].copy()

        # XGBoost necesită clase 0-indexed
        if model_name == "XGBoost" and y_val.min() == 1:
            y_val = y_val - 1

        #  Model A (Baseline) 
        path_A  = find_model_path(cfg["model_a_dir"], prefix, ext, fold_idx)
        pred_A  = load_and_predict(path_A, mtype, X_val)
        score_A = compute_metric(y_val, pred_A, metric)
        scores_A.append(score_A)
        preds_A_all.extend(pred_A)

        # Model B (TSTR)
        path_B  = find_model_path(cfg["model_b_dir"], prefix, ext, fold_idx)
        pred_B  = load_and_predict(path_B, mtype, X_val)
        score_B = compute_metric(y_val, pred_B, metric)
        scores_B.append(score_B)
        preds_B_all.extend(pred_B)

        y_true_all.extend(y_val)

        log.info(
            f"  Fold {fold_idx}: A={score_A:.4f}  B={score_B:.4f}  "
            f"Δ={score_B - score_A:+.4f}"
        )

    # Convertim la array
    scores_A   = np.array(scores_A)
    scores_B   = np.array(scores_B)
    diffs      = scores_B - scores_A          # Δk = B_k - A_k
    y_true_arr = np.array(y_true_all)
    preds_A_arr = np.array(preds_A_all)
    preds_B_arr = np.array(preds_B_all)
    M = len(diffs)

    
    #  1. STATISTICI DESCRIPTIVE
    
    mean_diff   = np.mean(diffs)
    median_diff = np.median(diffs)
    std_diff    = np.std(diffs, ddof=1)
    min_diff    = np.min(diffs)
    max_diff    = np.max(diffs)

    
    #  2. TEST WILCOXON SIGNED-RANK (H1: mediana(Δ) > 0)
    
    if np.all(diffs == 0):
        stat_w, p_wilcoxon = 0.0, 1.0
        z_wilcoxon = 0.0
        log.warning(f"  [{model_name}] Toate diferențele sunt zero — testul Wilcoxon nu e informativ.")
    else:
        stat_w, p_wilcoxon = wilcoxon(
            scores_B, scores_A,
            alternative="greater",   # H1: B > A
            zero_method="wilcox",    # exclude zero differences (clasic)
        )
        z_wilcoxon = wilcoxon_z_from_stat(stat_w, diffs)

    effect_r     = wilcoxon_effect_size(z_wilcoxon, M)
    effect_label = interpret_effect_size(effect_r)

    
    #  3. BOOTSTRAP CI (95%) — pentru medie și mediană
    
    _, ci_mean_lo, ci_mean_hi     = bootstrap_ci(
        diffs, stat_fn=np.mean,   n_boot=cfg["n_bootstrap"], seed=cfg["random_seed"]
    )
    _, ci_median_lo, ci_median_hi = bootstrap_ci(
        diffs, stat_fn=np.median, n_boot=cfg["n_bootstrap"], seed=cfg["random_seed"]
    )

    
    #  4. TEST McNEMAR (discordanțe pe predicții individuale)
    
    correct_A = (preds_A_arr == y_true_arr)
    correct_B = (preds_B_arr == y_true_arr)

    n11 = int(np.sum( correct_A &  correct_B))   # Ambele corecte
    n00 = int(np.sum(~correct_A & ~correct_B))   # Ambele greșite
    n10 = int(np.sum( correct_A & ~correct_B))   # Doar A corect
    n01 = int(np.sum(~correct_A &  correct_B))   # Doar B corect ("salvate de CVAE")

    contingency = [[n11, n10], [n01, n00]]
    mcn_result  = mcnemar(contingency, exact=False, correction=True)

    # McNemar manual 
    denom = n01 + n10
    if denom > 0:
        chi2_mcnemar = (abs(n01 - n10) - 1) ** 2 / denom   # Cu corecție Yates
    else:
        chi2_mcnemar = 0.0

    
    #  5. METRICI SUPLIMENTARE (Cohen's κ, F1-macro pe predicții totale)
    
    acc_A_total  = accuracy_score(y_true_arr, preds_A_arr)
    acc_B_total  = accuracy_score(y_true_arr, preds_B_arr)
    f1_A_total   = f1_score(y_true_arr, preds_A_arr, average="macro", zero_division=0)
    f1_B_total   = f1_score(y_true_arr, preds_B_arr, average="macro", zero_division=0)
    kappa_A      = cohen_kappa_score(y_true_arr, preds_A_arr)
    kappa_B      = cohen_kappa_score(y_true_arr, preds_B_arr)
    n_total      = len(y_true_arr)

    significant  = p_wilcoxon < cfg["alpha"]

    # Determinam rezultatul 
    result = {
        "model":         model_name,
        "M":             M,
        "n_total":       n_total,
        # Scoruri per fold
        "scores_A":      scores_A.tolist(),
        "scores_B":      scores_B.tolist(),
        "diffs":         diffs.tolist(),
        # Descriptive
        "mean_diff":     float(mean_diff),
        "median_diff":   float(median_diff),
        "std_diff":      float(std_diff),
        "min_diff":      float(min_diff),
        "max_diff":      float(max_diff),
        # Bootstrap CI
        "ci_mean":       [float(ci_mean_lo),   float(ci_mean_hi)],
        "ci_median":     [float(ci_median_lo), float(ci_median_hi)],
        # Wilcoxon
        "stat_W":        float(stat_w),
        "z_wilcoxon":    float(z_wilcoxon),
        "p_wilcoxon":    float(p_wilcoxon),
        "effect_r":      float(effect_r),
        "effect_label":  effect_label,
        "significant":   bool(significant),
        # McNemar
        "n11": n11, "n10": n10, "n01": n01, "n00": n00,
        "chi2_mcnemar":  float(chi2_mcnemar),
        "p_mcnemar":     float(mcn_result.pvalue),
        # Agregate totale
        "acc_A_total":   float(acc_A_total),
        "acc_B_total":   float(acc_B_total),
        "f1_A_total":    float(f1_A_total),
        "f1_B_total":    float(f1_B_total),
        "kappa_A":       float(kappa_A),
        "kappa_B":       float(kappa_B),
    }

    return result


In [35]:
#  RAPORTARE

def print_model_report(r: dict, metric_name: str = "Accuracy"):
    """Afișează raportul complet pentru un model în consolă."""
    sig_star = "SEMNIFICATIV" if r["significant"] else " NESEMNIFICATIV"
    M = r["M"]

    print_section(f"RAPORT STATISTIC — {r['model']}")

    # Per-fold 
    print_subsection(f"Per-fold {metric_name} (M = {M} fold-uri)")
    header = f"  {'Fold':>4}  {'Model A':>10}  {'Model B':>10}  {'Δ (B−A)':>12}"
    print(header)
    print("  " + "─" * (len(header) - 2))
    for k, (a, b, d) in enumerate(zip(r["scores_A"], r["scores_B"], r["diffs"])):
        sign = "+" if d >= 0 else ""
        print(f"  {k:>4}  {a:>10.4f}  {b:>10.4f}  {sign}{d:>11.4f}")

    print(f"\n  Mean A  = {np.mean(r['scores_A']):.4f} ± {np.std(r['scores_A'], ddof=1):.4f}")
    print(f"  Mean B  = {np.mean(r['scores_B']):.4f} ± {np.std(r['scores_B'], ddof=1):.4f}")

    #  Statistici diferențe 
    print_subsection("Statistici diferențe Δ = B − A")
    print(f"  Mediană Δ   : {r['median_diff']:+.4f}  ({r['median_diff']*100:+.2f}%)")
    print(f"  Medie Δ     : {r['mean_diff']:+.4f}  ({r['mean_diff']*100:+.2f}%)")
    print(f"  Std dev Δ   : {r['std_diff']:.4f}")
    print(f"  Min / Max Δ : {r['min_diff']:+.4f} / {r['max_diff']:+.4f}")
    print(f"  95% CI medie   : [{r['ci_mean'][0]:+.4f}, {r['ci_mean'][1]:+.4f}]  (bootstrap n={CFG['n_bootstrap']:,})")
    print(f"  95% CI mediană : [{r['ci_median'][0]:+.4f}, {r['ci_median'][1]:+.4f}]  (bootstrap n={CFG['n_bootstrap']:,})")

    # Test Wilcoxon
    print_subsection("Test Wilcoxon Signed-Rank  (H₁: mediana(Δ) > 0, one-sided)")
    print(f"  Perechi (M)   : {M}")
    print(f"  Statistică W  : {r['stat_W']:.4f}")
    print(f"  Z (approx.)   : {r['z_wilcoxon']:.4f}")
    print(f"  {format_pvalue(r['p_wilcoxon'])}")
    print(f"  Effect size r : {r['effect_r']:.3f}  → efect {r['effect_label']}")
    print(f"\n  Concluzie     : {sig_star}  (α = {CFG['alpha']})")

    #  Test McNemar 
    print_subsection(f"Test McNemar  (n = {r['n_total']:,} spectre totale)")
    print(f"  Tabelul de discordanțe:")
    print(f"  ┌─────────────────────────────────────┐")
    print(f"  │  Ambele corecte   (n₁₁) = {r['n11']:>6}    │")
    print(f"  │  Ambele greșite   (n₀₀) = {r['n00']:>6}    │")
    print(f"  │  Doar A corect    (n₁₀) = {r['n10']:>6}    │")
    print(f"  │  Doar B corect    (n₀₁) = {r['n01']:>6}    │  ← 'salvate de CVAE'")
    print(f"  └─────────────────────────────────────┘")
    print(f"  χ² (Yates)  : {r['chi2_mcnemar']:.4f}")
    print(f"  {format_pvalue(r['p_mcnemar'])}")

    #  Metrici agregate 
    print_subsection("Metrici agregate (predicții concatenate)")
    print(f"  {'Metrică':<22}  {'Model A':>10}  {'Model B':>10}  {'Δ':>10}")
    print("  " + "─" * 55)
    print(f"  {'Accuracy':<22}  {r['acc_A_total']:>10.4f}  {r['acc_B_total']:>10.4f}  {r['acc_B_total']-r['acc_A_total']:>+10.4f}")
    print(f"  {'F1-macro':<22}  {r['f1_A_total']:>10.4f}  {r['f1_B_total']:>10.4f}  {r['f1_B_total']-r['f1_A_total']:>+10.4f}")
    print(f"  {'Cohen κ':<22}  {r['kappa_A']:>10.4f}  {r['kappa_B']:>10.4f}  {r['kappa_B']-r['kappa_A']:>+10.4f}")

    #  Bloc text academic 
    print_subsection("Bloc text academic (copy-paste ready)")
    print(generate_academic_paragraph(r))


def generate_academic_paragraph(r: dict) -> str:
    sig_str = (
        f"a respins ipoteza nulă (Z = {r['z_wilcoxon']:.2f}, "
        f"{format_pvalue(r['p_wilcoxon'])})"
        if r["significant"]
        else (
            f"nu a respins ipoteza nulă (Z = {r['z_wilcoxon']:.2f}, "
            f"{format_pvalue(r['p_wilcoxon'])})"
        )
    )
    mcn_sig = (
        "semnificativă" if r["p_mcnemar"] < CFG["alpha"]
        else "nesemnificativă"
    )

    para = (
        f"  Analiza prin validare încrucișată ({r['M']} fold-uri) a evidențiat o creștere\n"
        f"  mediană a acurateței de {r['median_diff']:+.3f} ({r['median_diff']*100:+.1f}%) "
        f"în favoarea modelului augmentat (A: {np.mean(r['scores_A']):.3f}, "
        f"B: {np.mean(r['scores_B']):.3f}).\n"
        f"  Testul Wilcoxon pentru eșantioane pereche {sig_str}.\n"
        f"  Intervalul bootstrap 95% pentru media diferenței: "
        f"[{r['ci_mean'][0]:+.3f}, {r['ci_mean'][1]:+.3f}].\n"
        f"  Mărimea efectului r = {r['effect_r']:.2f} indică un efect {r['effect_label']}.\n"
        f"\n"
        f"  Analiza discordanțelor McNemar (χ² = {r['chi2_mcnemar']:.2f}, "
        f"{format_pvalue(r['p_mcnemar'])}) a indicat o diferență {mcn_sig}\n"
        f"  statistic între modele (n₀₁ = {r['n01']} spectre 'salvate' de CVAE, "
        f"n₁₀ = {r['n10']} spectre pierdute)."
    )
    return para


def print_summary_table(results: list[dict]):
    """Tabel comparativ final — toate modelele."""
    print_section("REZUMAT COMPARATIV — TOATE MODELELE")

    col_w = [18, 8, 10, 10, 8, 8, 10, 8, 10, 8]
    headers = [
        "Model", "M", "Med. Δ%", "Medie Δ%",
        "Z", "p-Wilcox", "Effect r", "n₀₁",
        "p-McNemar", "Semn.?"
    ]

    def row_str(vals):
        return "  " + "  ".join(
            str(v).ljust(w) for v, w in zip(vals, col_w)
        )

    print(row_str(headers))
    print("  " + "─" * (sum(col_w) + 2 * len(col_w)))

    for r in results:
        sig_mark = "✅" if r["significant"] else "❌"
        row = [
            r["model"],
            r["M"],
            f"{r['median_diff']*100:+.2f}%",
            f"{r['mean_diff']*100:+.2f}%",
            f"{r['z_wilcoxon']:.3f}",
            f"{r['p_wilcoxon']:.4f}",
            f"{r['effect_r']:.3f} ({r['effect_label'][:3]})",
            r["n01"],
            f"{r['p_mcnemar']:.4e}",
            sig_mark,
        ]
        print(row_str(row))

    print()



#  EXPORT REZULTATE


def save_results(results: list[dict], output_dir: str):
    """Salvează rezultatele în JSON și CSV."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    # JSON complet
    json_path = f"{output_dir}/eval_results_{ts}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    log.info(f"Rezultate JSON salvate: {json_path}")

    # CSV rezumat
    csv_rows = []
    for r in results:
        csv_rows.append({
            "model":          r["model"],
            "M":              r["M"],
            "n_total":        r["n_total"],
            "mean_A":         round(np.mean(r["scores_A"]), 4),
            "mean_B":         round(np.mean(r["scores_B"]), 4),
            "median_delta":   round(r["median_diff"], 4),
            "mean_delta":     round(r["mean_diff"], 4),
            "std_delta":      round(r["std_diff"], 4),
            "ci_mean_lo":     round(r["ci_mean"][0], 4),
            "ci_mean_hi":     round(r["ci_mean"][1], 4),
            "ci_median_lo":   round(r["ci_median"][0], 4),
            "ci_median_hi":   round(r["ci_median"][1], 4),
            "Z_wilcoxon":     round(r["z_wilcoxon"], 4),
            "p_wilcoxon":     round(r["p_wilcoxon"], 5),
            "effect_r":       round(r["effect_r"], 3),
            "effect_label":   r["effect_label"],
            "significant":    r["significant"],
            "n01":            r["n01"],
            "n10":            r["n10"],
            "n11":            r["n11"],
            "n00":            r["n00"],
            "chi2_mcnemar":   round(r["chi2_mcnemar"], 4),
            "p_mcnemar":      f"{r['p_mcnemar']:.4e}",
            "acc_A":          round(r["acc_A_total"], 4),
            "acc_B":          round(r["acc_B_total"], 4),
            "f1_macro_A":     round(r["f1_A_total"], 4),
            "f1_macro_B":     round(r["f1_B_total"], 4),
            "kappa_A":        round(r["kappa_A"], 4),
            "kappa_B":        round(r["kappa_B"], 4),
        })

    df = pd.DataFrame(csv_rows)
    csv_path = f"{output_dir}/eval_summary_{ts}.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    log.info(f"Rezumat CSV salvat: {csv_path}")

    return json_path, csv_path




In [37]:
def main():
    print_section("EVALUARE STATISTICĂ — AUGMENTARE DATE SINTETICE (CVAE)", width=72)
    print(f"  Data/ora   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Metrică    : {CFG['primary_metric']}")
    print(f"  Bootstrap  : n = {CFG['n_bootstrap']:,} iterații, seed = {CFG['random_seed']}")
    print(f"  α          : {CFG['alpha']}")
    print(f"  Modele     : {', '.join(CFG['models'].keys())}")

    #  Încărcare date
    log.info("Încărcăm setul de date...")
    with h5py.File(CFG["data_path"], "r") as f:
        X_all       = f["spectra"][:]
        y_all       = f["y_class"][:]
        samples_raw = f["sample_ids"][:]

    # Decodăm sample IDs dacă sunt bytes
    if isinstance(samples_raw[0], (bytes, np.bytes_)):
        samples_all = np.char.decode(samples_raw.astype("S"), "utf-8")
    else:
        samples_all = samples_raw

    log.info(f"  X: {X_all.shape}  |  y: {y_all.shape}  |  clase: {np.unique(y_all)}")

    with open(CFG["split_file"], "r") as f:
        splits = json.load(f)
    log.info(f"  Split-uri încărcate: {len(splits)} fold-uri")

    # Evaluare modele 
    all_results = []
    for model_name in CFG["models"]:
        try:
            result = evaluate_model(
                model_name, CFG, X_all, y_all, samples_all, splits
            )
            all_results.append(result)
            print_model_report(result, metric_name=CFG["primary_metric"].capitalize())
        except FileNotFoundError as e:
            log.error(f"Model '{model_name}' sărit — fișier lipsă: {e}")
        except Exception as e:
            log.error(f"Eroare la modelul '{model_name}': {e}", exc_info=True)

    #  Rezumat comparativ 
    if all_results:
        print_summary_table(all_results)
        json_path, csv_path = save_results(all_results, CFG["output_dir"])
        print(f"\n   Fișiere salvate:")
        print(f"     JSON : {json_path}")
        print(f"     CSV  : {csv_path}")

    print_section("EVALUARE COMPLETĂ", width=72)


if __name__ == "__main__":
    main()


20:25:25-INFO-Încărcăm setul de date...



════════════════════════════════════════════════════════════════════════
  EVALUARE STATISTICĂ — AUGMENTARE DATE SINTETICE (CVAE)
════════════════════════════════════════════════════════════════════════
  Data/ora   : 2026-03-16 20:25:25
  Metrică    : f1_macro
  Bootstrap  : n = 10,000 iterații, seed = 42
  α          : 0.05
  Modele     : Apollo, RF_Standard, XGBoost, 1D_CNN


20:25:35-INFO-  X: (277764, 1738)  |  y: (277764,)  |  clase: [0 1 2 3 4 5]
20:25:35-INFO-  Split-uri încărcate: 5 fold-uri
20:25:35-INFO-Evaluare model: Apollo
20:27:44-INFO-  Fold 0: A=0.2110  B=0.3191  Δ=+0.1082
20:30:30-INFO-  Fold 1: A=0.2369  B=0.4637  Δ=+0.2268
20:34:43-INFO-  Fold 2: A=0.2528  B=0.3135  Δ=+0.0607
20:37:42-INFO-  Fold 3: A=0.2243  B=0.3303  Δ=+0.1060
20:40:09-INFO-  Fold 4: A=0.2302  B=0.4644  Δ=+0.2342
20:40:09-INFO-Evaluare model: RF_Standard



══════════════════════════════════════════════════════════════════════
  RAPORT STATISTIC — Apollo
══════════════════════════════════════════════════════════════════════

  ────────────────────────────────────────────────────────────
  ▸ Per-fold F1_macro (M = 5 fold-uri)
  ────────────────────────────────────────────────────────────
  Fold     Model A     Model B       Δ (B−A)
  ──────────────────────────────────────────
     0      0.2110      0.3191  +     0.1082
     1      0.2369      0.4637  +     0.2268
     2      0.2528      0.3135  +     0.0607
     3      0.2243      0.3303  +     0.1060
     4      0.2302      0.4644  +     0.2342

  Mean A  = 0.2310 ± 0.0155
  Mean B  = 0.3782 ± 0.0786

  ────────────────────────────────────────────────────────────
  ▸ Statistici diferențe Δ = B − A
  ────────────────────────────────────────────────────────────
  Mediană Δ   : +0.1082  (+10.82%)
  Medie Δ     : +0.1472  (+14.72%)
  Std dev Δ   : 0.0784
  Min / Max Δ : +0.0607 / +0.2342
  

20:40:10-INFO-  Fold 0: A=0.2000  B=0.3390  Δ=+0.1390
20:40:12-INFO-  Fold 1: A=0.2251  B=0.4264  Δ=+0.2013
20:40:13-INFO-  Fold 2: A=0.1563  B=0.3647  Δ=+0.2084
20:40:14-INFO-  Fold 3: A=0.1757  B=0.2114  Δ=+0.0357
20:40:16-INFO-  Fold 4: A=0.2944  B=0.4002  Δ=+0.1058
20:40:16-INFO-Evaluare model: XGBoost



══════════════════════════════════════════════════════════════════════
  RAPORT STATISTIC — RF_Standard
══════════════════════════════════════════════════════════════════════

  ────────────────────────────────────────────────────────────
  ▸ Per-fold F1_macro (M = 5 fold-uri)
  ────────────────────────────────────────────────────────────
  Fold     Model A     Model B       Δ (B−A)
  ──────────────────────────────────────────
     0      0.2000      0.3390  +     0.1390
     1      0.2251      0.4264  +     0.2013
     2      0.1563      0.3647  +     0.2084
     3      0.1757      0.2114  +     0.0357
     4      0.2944      0.4002  +     0.1058

  Mean A  = 0.2103 ± 0.0536
  Mean B  = 0.3483 ± 0.0835

  ────────────────────────────────────────────────────────────
  ▸ Statistici diferențe Δ = B − A
  ────────────────────────────────────────────────────────────
  Mediană Δ   : +0.1390  (+13.90%)
  Medie Δ     : +0.1380  (+13.80%)
  Std dev Δ   : 0.0715
  Min / Max Δ : +0.0357 / +0.20

20:40:17-INFO-  Fold 0: A=0.2443  B=0.3709  Δ=+0.1265
20:40:18-INFO-  Fold 1: A=0.1967  B=0.4163  Δ=+0.2196
20:40:19-INFO-  Fold 2: A=0.1790  B=0.4047  Δ=+0.2257
20:40:20-INFO-  Fold 3: A=0.2179  B=0.2458  Δ=+0.0279
20:40:21-INFO-  Fold 4: A=0.3084  B=0.3734  Δ=+0.0649
20:40:21-INFO-Evaluare model: 1D_CNN
20:40:22-ERROR-Eroare la modelul '1D_CNN': Unable to synchronously open file (file signature not found)
Traceback (most recent call last):
  File "C:\Users\Iulian\AppData\Local\Temp\ipykernel_2340\4074182394.py", line 32, in main
    result = evaluate_model(
  File "C:\Users\Iulian\AppData\Local\Temp\ipykernel_2340\4093979367.py", line 45, in evaluate_model
    pred_A  = load_and_predict(path_A, mtype, X_val)
  File "C:\Users\Iulian\AppData\Local\Temp\ipykernel_2340\1427606014.py", line 157, in load_and_predict
    with h5py.File(temp_model_path, 'r+') as f:
  File "c:\Users\Iulian\anaconda3\envs\myenv\lib\site-packages\h5py\_hl\files.py", line 566, in __init__
    fid = make_fid(name


══════════════════════════════════════════════════════════════════════
  RAPORT STATISTIC — XGBoost
══════════════════════════════════════════════════════════════════════

  ────────────────────────────────────────────────────────────
  ▸ Per-fold F1_macro (M = 5 fold-uri)
  ────────────────────────────────────────────────────────────
  Fold     Model A     Model B       Δ (B−A)
  ──────────────────────────────────────────
     0      0.2443      0.3709  +     0.1265
     1      0.1967      0.4163  +     0.2196
     2      0.1790      0.4047  +     0.2257
     3      0.2179      0.2458  +     0.0279
     4      0.3084      0.3734  +     0.0649

  Mean A  = 0.2293 ± 0.0505
  Mean B  = 0.3622 ± 0.0679

  ────────────────────────────────────────────────────────────
  ▸ Statistici diferențe Δ = B − A
  ────────────────────────────────────────────────────────────
  Mediană Δ   : +0.1265  (+12.65%)
  Medie Δ     : +0.1329  (+13.29%)
  Std dev Δ   : 0.0892
  Min / Max Δ : +0.0279 / +0.2257
 

20:40:22-INFO-Rezultate JSON salvate: ../results/statistical_eval/eval_results_20260316_204022.json
20:40:22-INFO-Rezumat CSV salvat: ../results/statistical_eval/eval_summary_20260316_204022.csv



   Fișiere salvate:
     JSON : ../results/statistical_eval/eval_results_20260316_204022.json
     CSV  : ../results/statistical_eval/eval_summary_20260316_204022.csv

════════════════════════════════════════════════════════════════════════
  EVALUARE COMPLETĂ
════════════════════════════════════════════════════════════════════════
